In [ ]:
from email import charset
import math
import random
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.backend import get_value, ctc_decode
from keras import Model
from tensorflow.keras import backend as K
import tensorflow as tf
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.layers import Input, Conv2D, SeparableConv2D, MaxPooling2D, AveragePooling2D
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, LeakyReLU, Reshape, Bidirectional, LSTM, GRU, LayerNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Nadam
import os
import numpy as np
import cv2
import string

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nibinv23/iam-handwriting-word-database")

print("Path to dataset files:", path)

In [ ]:
import kagglehub
import os

# Скачиваем датасет
path = kagglehub.dataset_download("nibinv23/iam-handwriting-word-database")
print("Путь к датасету:", path)

# Проверяем содержимое папки
print("Файлы в датасете:", os.listdir(path))

In [ ]:
word_dict = {}

with open("/root/.cache/kagglehub/datasets/nibinv23/iam-handwriting-word-database/versions/2/words_new.txt", "r", encoding="utf-8") as file:
    for line in file:
        # Пропускаем строки, начинающиеся с #
        if line.startswith("#"):
            continue

        parts = line.strip().split()

        # Пропускаем строки с ошибками сегментации
        if parts[1] != "ok":
            continue

        # Берём первый элемент как ключ и последний как значение
        word_id = parts[0]
        word = parts[-1]

        word_dict[word_id] = word


In [ ]:
# Проверим, что получилось
print(list(word_dict.items())[:100])  # Выведет первые 10 элементов словаря

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def adaptive_threshold(image):
    # Применим фильтр Гаусса
    blurred = cv2.GaussianBlur(image, (5, 5), 0)

    # Применение адаптивной бинаризации
    binary = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                 cv2.THRESH_BINARY, 21, 10)

    # Инверсия цветов
    inverted = cv2.bitwise_not(binary)

    return inverted

def normalize(image):
    normalized = image.astype(np.float32) / 255.0

    return normalized

def resize_and_reshape(image, target_size=(64, 200)):
    # Конвертация в grayscale
    if len(image.shape) == 3:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    else:
        gray = image.copy()

    # Изменение размера
    h, w = gray.shape
    if h > target_size[0] or w > target_size[1]:
        shrink = min(target_size[0]/h, target_size[1]/w)
        resized = cv2.resize(gray, None, fx=shrink, fy=shrink,
                            interpolation=cv2.INTER_AREA)
    else:
        resized = gray.copy()

    # Добавление рамок
    pad_h = target_size[0] - resized.shape[0]
    pad_w = target_size[1] - resized.shape[1]
    padded = cv2.copyMakeBorder(resized,
                               math.ceil(pad_h/2), math.floor(pad_h/2),
                               math.ceil(pad_w/2), math.floor(pad_w/2),
                               cv2.BORDER_CONSTANT, value=255)

    # Поворот
    rotated = cv2.rotate(padded, cv2.ROTATE_90_CLOCKWISE)

    return rotated

def preprocess_image(image):
    resized = resize_and_reshape(image)

    thresholded = adaptive_threshold(resized)

    normalized = normalize(thresholded)

    return normalized

In [ ]:
import tensorflow as tf
from tensorflow import keras

class CharacterErrorRateMaxLen(keras.metrics.Metric):
    def __init__(self, pad_token=0, name="character_error_score", **kwargs):
        super().__init__(name=name, **kwargs)
        self.pad_token = pad_token

        self.error_sum = self.add_weight(
            name="cumulative_errors", initializer="zeros"
        )
        self.num_samples = self.add_weight(
            name="sample_counter", initializer="zeros"
        )

    def update_state(self, y_true, y_pred, sample_weight=None):

        y_pred_ids = tf.argmax(y_pred, axis=-1, output_type=tf.int32)

        mask = tf.not_equal(y_true, self.pad_token)

        y_true_masked = tf.where(mask, y_true, 0)
        y_pred_masked = tf.where(mask, y_pred_ids, 0)

        sparse_true = tf.sparse.from_dense(y_true_masked)
        sparse_pred = tf.sparse.from_dense(y_pred_masked)

        error_rate = tf.edit_distance(
            sparse_pred,
            sparse_true,
            normalize=True
        )

        self.error_sum.assign_add(tf.reduce_sum(error_rate))

        batch_size = tf.cast(tf.shape(y_true)[0], tf.float32)
        self.num_samples.assign_add(batch_size)

    def result(self):
        return tf.math.divide_no_nan(self.error_sum, self.num_samples)

    def reset_state(self):
        self.error_sum.assign(0.0)
        self.num_samples.assign(0.0)

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

# Загружаем модель
model_path = 'sample_data/char_lm_decoder1.keras'
loaded_model = keras.models.load_model(model_path)

print("Модель успешно загружена!")
loaded_model.summary()

In [ ]:
class CharacterErrorRate(tf.keras.metrics.Metric):
    """
    CER (Character Error Rate) для decoder модели с teacher forcing.
    """
    def __init__(self, name='character_error_rate', pad_token=0, eos_token=2, **kwargs):
        super().__init__(name=name, **kwargs)
        self.pad_token = pad_token
        self.eos_token = eos_token
        self.error_sum = self.add_weight(name="cumulative_errors", initializer="zeros")
        self.char_count = self.add_weight(name="total_characters", initializer="zeros")
        self.sample_count = self.add_weight(name="sample_counter", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):

        y_true = tf.cast(y_true, tf.int32)

        y_pred_tokens = tf.argmax(y_pred, axis=-1, output_type=tf.int32)

        mask = tf.cast(tf.not_equal(y_true, self.pad_token), tf.float32)

        batch_size = tf.shape(y_true)[0]
        seq_len = tf.shape(y_true)[1]

        eos_positions = tf.argmax(tf.cast(tf.equal(y_true, self.eos_token), tf.int32), axis=1)
        eos_positions = tf.cast(eos_positions, tf.int32)

        range_tensor = tf.range(seq_len, dtype=tf.int32)
        range_tensor = tf.expand_dims(range_tensor, 0)
        range_tensor = tf.tile(range_tensor, [batch_size, 1])

        eos_positions_expanded = tf.expand_dims(eos_positions, 1)

        before_eos_mask = tf.cast(tf.less_equal(range_tensor, eos_positions_expanded), tf.float32)

        final_mask = mask * before_eos_mask

        chars_per_sample = tf.reduce_sum(final_mask, axis=1)

        not_equal = tf.cast(tf.not_equal(y_true, y_pred_tokens), tf.float32)
        masked_errors = not_equal * final_mask
        batch_errors = tf.reduce_sum(masked_errors)

        self.error_sum.assign_add(batch_errors)
        self.char_count.assign_add(tf.reduce_sum(chars_per_sample))
        self.sample_count.assign_add(tf.cast(batch_size, tf.float32))

    def result(self):
        return tf.math.divide_no_nan(self.error_sum, self.char_count)

    def reset_state(self):
        self.error_sum.assign(0.0)
        self.char_count.assign(0.0)
        self.sample_count.assign(0.0)

In [ ]:
import tensorflow as tf
import keras_nlp
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import AdamW, Nadam

def create_ocr_with_loaded_decoder(
    num_classes,
    decoder_model_path,
    input_shape=(200, 64, 1),
    max_decoder_len=20
):

    pretrained_decoder = keras.models.load_model(decoder_model_path)

    image_inputs = keras.Input(shape=input_shape, name="image_input")

    x = layers.Concatenate()([image_inputs, image_inputs, image_inputs])

    base_model = VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=(input_shape[0], input_shape[1], 3)
    )

    output_layer = base_model.get_layer("block3_pool").output
    vgg_truncated = keras.Model(base_model.input, output_layer)

    for layer in vgg_truncated.layers:
        if "block1" in layer.name or "block2" in layer.name:
            layer.trainable = False
        else:
            layer.trainable = True

    x = vgg_truncated(x)

    x = layers.Conv2D(256, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)

    x = layers.Conv2D(256, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.1)(x)

    h, w, c = x.shape[1], x.shape[2], x.shape[3]
    x = layers.Reshape((w, h * c))(x)

    x = Dense(256)(x)
    x = LeakyReLU(0.01)(x)
    x = Dropout(0.5)(x)
    encoder_outputs = layers.Dense(256, name="encoder_projection")(x)

    decoder_inputs = keras.Input(
        shape=(max_decoder_len - 1,),
        dtype=tf.int32,
        name="decoder_input_tokens"
    )

    embedding_layer = pretrained_decoder.layers[1]
    decoder_embeddings = embedding_layer(decoder_inputs)

    lm_transformer = pretrained_decoder.layers[2]

    new_transformer = keras_nlp.layers.TransformerDecoder(
        intermediate_dim=lm_transformer.intermediate_dim,
        num_heads=lm_transformer.num_heads,
        dropout=lm_transformer.dropout,
        activation=lm_transformer.activation,
        normalize_first=lm_transformer.normalize_first,
    )

    decoder_output = new_transformer(
        decoder_sequence=decoder_embeddings,
        encoder_sequence=encoder_outputs,
    )

    for w_new, w_old in zip(new_transformer.weights, lm_transformer.weights):
        if w_new.shape == w_old.shape:
            w_new.assign(w_old)

    output_layer = pretrained_decoder.layers[-1]
    outputs = output_layer(decoder_output)

    model = keras.Model(
        inputs=[image_inputs, decoder_inputs],
        outputs=outputs,
        name="ocr_model"
    )
    cer_metric = CharacterErrorRate(
    pad_token=char_to_idx['<pad>'])

    model.compile(
        optimizer=AdamW(learning_rate=3e-4, weight_decay=0.01),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        weighted_metrics=[cer_metric],
    )

    return model

In [ ]:
def process_data(word_dict):
    image_base_folder = f'/root/.cache/kagglehub/datasets/nibinv23/iam-handwriting-word-database/versions/2/iam_words/words'
    train_X, train_y = [], []
    total_count = 0
    folder_counts = {}
    max_per_folder = 200
    max_total = 30000

    for root, _, files in os.walk(image_base_folder):
        if total_count >= max_total:
            break

        folder_name = os.path.basename(root)
        folder_counts[folder_name] = 0

        for image_file in files:
            if total_count >= max_total:
                break

            if folder_counts[folder_name] >= max_per_folder:
                break

            if image_file.endswith('.jpg') or image_file.endswith('.png'):
                image_name = image_file.rsplit('.', 1)[0]

                # Проверяем, есть ли имя в словаре
                if image_name not in word_dict:
                    continue  # Пропускаем изображение без метки

                label = word_dict[image_name]
                if label == 'Ps':
                  continue

                if not any(c not in string.ascii_letters + ',.\' ' for c in label):
                    image_path = os.path.join(root, image_file)
                    if os.path.exists(image_path):
                        try:
                            img_array = cv2.imread(image_path, cv2.IMREAD_COLOR)
                            if img_array is None:
                                raise ValueError("Failed to read image with OpenCV")

                            if len(label) <= 12:
                                img_array = cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB)
                                train_X.append(preprocess_image(img_array))
                                train_y.append(label)
                                folder_counts[folder_name] += 1
                                total_count += 1
                        except Exception as e:
                            print(f"Failed to load image: {image_path}, error: {e}")
                    else:
                        print(f"File does not exist: {image_path}")

    return np.array(train_X), np.array(train_y)


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt


def create_token_mappings(texts):

    all_chars = set(''.join(texts))
    alphabet = sorted(list(all_chars))

    special_tokens = ['<pad>', '<bos>', '<eos>', '<unk>']

    idx_to_char = {i: token for i, token in enumerate(special_tokens)}
    idx_to_char.update({i+4: c for i, c in enumerate(alphabet)})

    char_to_idx = {v: k for k, v in idx_to_char.items()}

    vocab_size = len(special_tokens) + len(alphabet)

    print(f"Символов в алфавите: {len(alphabet)}")
    print(f"Всего токенов (включая специальные): {vocab_size}")
    print("Специальные токены:", special_tokens)

    return char_to_idx, idx_to_char, vocab_size

def encode_text(
    text,
    char_to_idx,
    max_len=20,
    bos_token=1,
    eos_token=2,
    pad_token=0,
    unk_token=3
):

    indices = [char_to_idx.get(c, unk_token) for c in text]

    sequence = [bos_token] + indices + [eos_token]

    if len(sequence) > max_len:
        sequence = sequence[:max_len-1] + [eos_token]

    padded = sequence + [pad_token] * (max_len - len(sequence))

    return np.array(padded, dtype=np.int32)

def prepare_data_for_decoder(
    images,
    texts,
    max_len=20,
    train_split=0.7,
    val_split=0.15,
    test_split=0.15,
    shuffle=True,
    seed=42
):

    char_to_idx, idx_to_char, vocab_size = create_token_mappings(texts)

    encoded_labels = np.array([
        encode_text(
            text,
            char_to_idx,
            max_len=max_len,
            bos_token=char_to_idx['<bos>'],
            eos_token=char_to_idx['<eos>'],
            pad_token=char_to_idx['<pad>'],
            unk_token=char_to_idx['<unk>']
        )
        for text in texts
    ])

    n = len(images)
    indices = np.arange(n)

    if shuffle:
        np.random.seed(seed)
        np.random.shuffle(indices)

    train_end = int(n * train_split)
    val_end = train_end + int(n * val_split)

    train_idx = indices[:train_end]
    val_idx   = indices[train_end:val_end]
    test_idx  = indices[val_end:]

    train_images = images[train_idx]
    train_labels = encoded_labels[train_idx]
    train_input  = train_labels[:, :-1]
    train_target = train_labels[:, 1:]

    val_images = images[val_idx]
    val_labels = encoded_labels[val_idx]
    val_input  = val_labels[:, :-1]
    val_target = val_labels[:, 1:]

    test_images = images[test_idx]
    test_labels = encoded_labels[test_idx]
    test_input  = test_labels[:, :-1]
    test_target = test_labels[:, 1:]

    train_ds = tf.data.Dataset.from_tensor_slices(
        ((train_images, train_input), train_target)
    ).batch(32).prefetch(tf.data.AUTOTUNE)

    val_ds = tf.data.Dataset.from_tensor_slices(
        ((val_images, val_input), val_target)
    ).batch(32).prefetch(tf.data.AUTOTUNE)

    test_ds = tf.data.Dataset.from_tensor_slices(
        ((test_images, test_input), test_target)
    ).batch(32).prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds, test_ds, char_to_idx, idx_to_char, vocab_size


def visualize_dataset_samples(dataset, idx_to_char, dataset_name="", num_samples=10):

    batch_count = 0
    total_shown = 0

    for (image_batch, decoder_input_batch), target_batch in dataset.take(3):
        batch_size = image_batch[0].shape[0]

        for i in range(min(batch_size, num_samples - total_shown)):
            if total_shown >= num_samples:
                break

            image = image_batch[0][i].numpy()
            decoder_input = decoder_input_batch[i].numpy()
            target = target_batch[i].numpy()

            decoder_text = decode_tokens(decoder_input, idx_to_char, is_decoder_input=True)

            target_text = decode_tokens(target, idx_to_char, is_target=True)

            if total_shown < 5:
                plt.figure(figsize=(10, 4))

                plt.subplot(1, 2, 1)
                plt.imshow(image.squeeze(), cmap='gray')
                plt.title(f'Изображение {total_shown + 1}')
                plt.axis('off')

                plt.subplot(1, 2, 2)
                plt.text(0.1, 0.7, f"Decoder input:\n{decoder_text}",
                        fontsize=12, va='top', fontweight='bold')
                plt.text(0.1, 0.4, f"Target (что предсказывать):\n{target_text}",
                        fontsize=12, va='top', fontweight='bold', color='green')
                plt.axis('off')
                plt.tight_layout()
                plt.show()

            total_shown += 1

        batch_count += 1
        if total_shown >= num_samples:
            break


def decode_tokens(tokens, idx_to_char, is_decoder_input=False, is_target=False):
    text = ''

    for token in tokens:
        token = int(token)

        if token == 0:
            continue

        if is_decoder_input:
            if token == 1:
                text += '<bos>'
                continue

        if is_target:
            if token == 2:
                text += '<eos>'
                break

        char = idx_to_char.get(token, f'?{token}?')

        if is_decoder_input and char in ['<eos>', '<unk>']:
            continue

        text += char

    return text

In [ ]:
train_X, train_y = process_data(word_dict)

In [ ]:
train_X, train_y

In [ ]:
train_ds, val_ds, test_ds, char_to_idx, idx_to_char, vocab_size = prepare_data_for_decoder(
        images=train_X,
        texts=train_y,
        max_len=20,
        train_split=0.8,
        val_split=0.1,
        test_split=0.1,
        shuffle=True,
        seed=42
    )

In [ ]:
model = create_ocr_with_loaded_decoder(
    num_classes=vocab_size,
    input_shape=(200, 64, 1),
    decoder_model_path='sample_data/char_lm_decoder1.keras',
)

In [ ]:
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
early_stopping = EarlyStopping(
    monitor='val_cer',
    patience=10,
    restore_best_weights=True,
    mode='min'
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    min_lr=1e-5,
    patience=4,
    mode='min'
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,
    callbacks=[early_stopping, reduce_lr],
)

In [ ]:
from google.colab import files
model.save('handwritten_decode_vgg6_last2.keras')
files.download('handwritten_decode_vgg6_last2.keras')

In [ ]:
loss, accuracy = model.evaluate(test_ds)

print(f'Loss: {loss}')
print(f'Accuracy: {accuracy}')

In [ ]:
fig, axes = plt.subplots(figsize=(12, 6), ncols=2, nrows=1)

axes[0].plot(history.history['loss'], label='loss')
axes[0].plot(history.history['val_loss'], label='val_loss')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[1].plot(history.history['cer'], label='cer')
axes[1].plot(history.history['val_cer'], label='val_cer')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Character Error Score")
axes[1].legend()

plt.show()